In [ ]:
!rm -rf '/content/DIS_Hughen'
!git clone https://github.com/NU-Academics/DIS_Hughen.git

In [1]:
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from xgboost import XGBClassifier

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
df = pd.read_csv("undersampled_CIC2019_dataset.csv")
# df = pd.read_csv("/content/DIS_Hughen/undersampled_CIC2019_dataset.csv")

In [5]:
df.shape

(4685611, 90)

In [6]:
df_sample = (df.groupby("label", group_keys=False).sample(n=2000, random_state=42, replace=True).reset_index(drop=True))

In [7]:
df_sample.shape

(38000, 90)

In [8]:
le = LabelEncoder()
df_sample["label"] = le.fit_transform(df_sample["label"])
X = df_sample.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df_sample["label"]
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(label_mapping)

{'BENIGN': 0, 'DrDoS_DNS': 1, 'DrDoS_LDAP': 2, 'DrDoS_MSSQL': 3, 'DrDoS_NTP': 4, 'DrDoS_NetBIOS': 5, 'DrDoS_SNMP': 6, 'DrDoS_SSDP': 7, 'DrDoS_UDP': 8, 'LDAP': 9, 'MSSQL': 10, 'NetBIOS': 11, 'Portmap': 12, 'Syn': 13, 'TFTP': 14, 'UDP': 15, 'UDP-lag': 16, 'UDPLag': 17, 'WebDDoS': 18}


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

In [10]:
class CICDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CICDataset(X_train, y_train)
train_loader = DataLoader(train_dataset,
                          batch_size=64, #8192
                          shuffle=True,
                          pin_memory=True,
                          num_workers=0) #4

In [11]:
import torch.nn as nn
import torch.nn.functional as F

class DNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 64)

        self.output = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        features = F.relu(self.fc3(x))
        logits = self.output(features)
        return logits, features

In [12]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

In [13]:
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,        # 10% validation
    stratify=y_train,     # important for imbalance
    random_state=42
)

model = DNN(input_dim=X_train.shape[1],
            num_classes=len(np.unique(y_train))).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience = 3
counter = 0

for epoch in range(30):  # max epochs
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits, _ = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.tensor(X_val.values,
                                    dtype=torch.float32).to(device)
        y_val_tensor = torch.tensor(y_val.values,
                                    dtype=torch.long).to(device)

        val_logits, _ = model(X_val_tensor)
        val_loss = criterion(val_logits, y_val_tensor)

    print(f"Epoch {epoch+1} | Train Loss: {total_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_dnn.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1 | Train Loss: 809.7959 | Val Loss: 1.1762
Epoch 2 | Train Loss: 612.1965 | Val Loss: 1.0246
Epoch 3 | Train Loss: 563.4216 | Val Loss: 0.9866
Epoch 4 | Train Loss: 537.9975 | Val Loss: 0.9928
Epoch 5 | Train Loss: 526.6966 | Val Loss: 0.9580
Epoch 6 | Train Loss: 507.7321 | Val Loss: 0.9719
Epoch 7 | Train Loss: 504.7492 | Val Loss: 0.9485
Epoch 8 | Train Loss: 501.2873 | Val Loss: 0.9159
Epoch 9 | Train Loss: 499.5044 | Val Loss: 0.9826
Epoch 10 | Train Loss: 489.1066 | Val Loss: 1.0822
Epoch 11 | Train Loss: 490.2901 | Val Loss: 0.9383
Early stopping triggered.


In [14]:
def extract_features(model, X, batch_size=8192):
    model.eval()
    dataset = torch.tensor(X.values, dtype=torch.float32)
    loader = DataLoader(dataset, batch_size=batch_size)

    features_list = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, features = model(batch)
            features_list.append(features.cpu().numpy())

    return np.vstack(features_list)

Z_train = extract_features(model, X_train)
Z_test = extract_features(model, X_test)

In [15]:
X_train_hybrid = np.hstack([X_train.values, Z_train])
X_test_hybrid = np.hstack([X_test.values, Z_test])

In [17]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
#   device='cuda',
    objective='multi:softprob',
    eval_metric='mlogloss'
)

param_dist = {
    "max_depth": [6, 7, 8, 9],
    "learning_rate": [0.03, 0.05, 0.07],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.2]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    verbose=2
)

search.fit(X_train_hybrid, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.7; total time=  48.7s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.7; total time=  50.8s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.7; total time=  47.7s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.03, max_depth=7, min_child_weight=1, subsample=0.7; total time=  45.7s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.03, max_depth=7, min_child_weight=1, subsample=0.7; total time=  46.2s
[CV] END colsample_bytree=0.7, gamma=0.2, learning_rate=0.03, max_depth=7, min_child_weight=1, subsample=0.7; total time=  57.4s
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.07, max_depth=8, min_child_weight=5, subsample=0.8; total time=  49.2s
[CV] END colsample_bytree=0.8, gamma

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [6, 7, ...], ...}"
,n_iter,20
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [18]:
y_pred = search.predict(X_test_hybrid)
y_prob = search.predict_proba(X_test_hybrid)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted"))

Accuracy: 0.7375
Weighted F1: 0.7324059619619517
Macro F1: 0.7324059619619517
ROC-AUC: 0.9803159265350876


In [19]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

               precision    recall  f1-score   support

       BENIGN       1.00      1.00      1.00       400
    DrDoS_DNS       0.60      0.66      0.63       400
   DrDoS_LDAP       0.46      0.41      0.43       400
  DrDoS_MSSQL       0.62      0.64      0.63       400
    DrDoS_NTP       1.00      1.00      1.00       400
DrDoS_NetBIOS       0.94      0.78      0.85       400
   DrDoS_SNMP       0.66      0.34      0.45       400
   DrDoS_SSDP       0.48      0.39      0.43       400
    DrDoS_UDP       0.40      0.37      0.38       400
         LDAP       0.46      0.72      0.56       400
        MSSQL       0.65      0.63      0.64       400
      NetBIOS       0.89      0.87      0.88       400
      Portmap       0.81      0.95      0.88       400
          Syn       1.00      1.00      1.00       400
         TFTP       1.00      0.99      1.00       400
          UDP       0.43      0.38      0.40       400
      UDP-lag       0.66      0.88      0.76       400
       UD